# Hierarchical Architecture — LangChain

**Pattern:** Manager → Team Leads → Workers  
**Framework:** LangChain LCEL  

```
Manager
  ├── Research Lead
  │     ├── weather_worker
  │     └── safety_worker
  └── Report Lead
        ├── format_worker
        └── summary_worker
```

**LangChain approach:** Hierarchy via Python function calls.  
Each 'lead' is a Python function that calls its worker chains.

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()
print("✓ Ready")

In [ ]:
WEATHER_DATA = {
    "tokyo": "Clear, 18°C, score 9/10",
    "paris": "Partly Cloudy, 16°C, score 7/10",
    "bangalore": "Rainy, 25°C, score 6/10",
}
SAFETY_DATA = {
    "tokyo": "Low advisory, score 10/10",
    "paris": "Low advisory, score 8/10",
    "bangalore": "Medium advisory, score 6/10",
}

# Worker chains
weather_worker = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a 1-sentence weather report."),
        ("human", "City: {city}\nData: {data}"),
    ]) | llm | parser
)
safety_worker = (
    ChatPromptTemplate.from_messages([
        ("system", "Write a 1-sentence safety report."),
        ("human", "City: {city}\nData: {data}"),
    ]) | llm | parser
)
format_worker = (
    ChatPromptTemplate.from_messages([
        ("system", "Format into a polished markdown section."),
        ("human", "City: {city}\nResearch: {research}\n\nUse '### {city}' header."),
    ]) | llm | parser
)
manager_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "Write an executive summary with top recommendation."),
        ("human", "Cities: {cities}\nReports:\n{reports}"),
    ]) | llm | parser
)
print("Worker chains ready")

In [ ]:
def research_lead(city: str) -> str:
    """Research lead: delegates to weather + safety workers."""
    weather = weather_worker.invoke({"city": city, "data": WEATHER_DATA.get(city.lower(), "N/A")})
    safety = safety_worker.invoke({"city": city, "data": SAFETY_DATA.get(city.lower(), "N/A")})
    print(f"  [Research Lead] {city}: weather ✓ | safety ✓")
    return f"Weather: {weather}\nSafety: {safety}"

def run_hierarchical(cities: list) -> str:
    all_reports = []
    for city in cities:
        print(f"\n[Manager → Research Lead] {city}")
        research = research_lead(city)
        print(f"[Manager → Report Lead] {city}")
        formatted = format_worker.invoke({"city": city, "research": research})
        all_reports.append(formatted)
    combined = "\n\n".join(all_reports)
    print("\n[Manager] executive summary...")
    summary = manager_chain.invoke({"cities": ", ".join(cities), "reports": combined})
    return f"# Hierarchical Report\n\n{combined}\n\n---\n\n## Executive Summary\n{summary}"

report = run_hierarchical(["Tokyo", "Paris"])
print("\n" + "="*50)
print(report)

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| Python functions as 'team leads' | Hierarchy = function call nesting in LangChain |
| Manager synthesizes after leads complete | Manager chain sees all results, writes summary |
| Workers are stateless chains | Each worker only sees its own inputs |

**LangChain limitation:** Hierarchy is implicit in code structure, not visible in any diagram.

### Next: [LangGraph Hierarchical →](../LangGraph/hierarchical.ipynb)